# Geopolitical Exposure Proof of Concept
## 04 — Locked Industry Benchmark and Event-Study Evaluation

This notebook opens the outcome quarantine only after verifying the frozen text seeds and expansion artifacts. It acquires point-in-time industry identifiers and daily returns, constructs market-model cumulative abnormal returns (CARs), and compares text, holdings embeddings, characteristics, industry, and shuffled-seed placebos.

The 2021Q4 holdings snapshot is treated as a researcher-information proof of concept. Because exact public-release dates have not been verified for every S12 fund, results must not be described as a tradable strategy.

## 1. Environment and frozen-input verification
No WRDS query or event return is loaded until every recorded expansion hash is verified.

In [ ]:
from pathlib import Path
import getpass, hashlib, importlib, json, sys, time
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, ttest_ind
import statsmodels.api as sm

ROOT = Path.cwd()
if ROOT.name == 'geopolitical_exposure':
    GEO_ROOT, ROOT = ROOT, ROOT.parent
else:
    GEO_ROOT = ROOT / 'geopolitical_exposure'
if not GEO_ROOT.exists():
    raise FileNotFoundError('Open from Asset_Embeddings or geopolitical_exposure.')
sys.path.insert(0, str(ROOT / '.packages'))
sys.path.insert(0, str(GEO_ROOT / 'src'))
import event_study
importlib.reload(event_study)
from event_study import value_weight_firm_returns, market_model_cars, industry_seed_scores

EXPANSION = GEO_ROOT / 'embedding_expansion'
MANIFESTS = GEO_ROOT / 'manifests'
EVENT_DATA = GEO_ROOT / 'event_data'
EVENT_RESULTS = GEO_ROOT / 'event_study'
EVENT_DATA.mkdir(parents=True, exist_ok=True)
EVENT_RESULTS.mkdir(parents=True, exist_ok=True)
freeze = json.loads((MANIFESTS / 'embedding_expansion_freeze_v1.json').read_text())
checks = []
for artifact in freeze['artifacts']:
    path = EXPANSION / artifact['path']
    checks.append({'artifact': artifact['artifact'], 'exists': path.exists(),
                   'hash_matches': path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == artifact['sha256'],
                   'bytes_match': path.exists() and path.stat().st_size == artifact['bytes']})
freeze_check = pd.DataFrame(checks)
display(freeze_check)
assert not freeze['outcomes_loaded'] and freeze_check[['exists','hash_matches','bytes_match']].all().all()


,artifact,exists,hash_matches,bytes_match
0,text_only_scores,True,True,True
1,embedding_propagated_scores,True,True,True
2,characteristic_propagated_scores,True,True,True
3,shuffled_embedding_placebos_100,True,True,True
4,top_20_nonseed_candidates,True,True,True
5,representation_rank_correlations,True,True,True


## 2. Freeze the evaluation protocol before opening outcomes
The event date, estimation window, CAR windows, minimum observations, exclusions, comparisons, and interpretation are written to a hashed manifest first.

In [ ]:
protocol = {
    'event_id': 'russia_ukraine_invasion_2022', 'event_date': '2022-02-24',
    'embedding_snapshot': '2021Q4', 'estimation_window': [-250, -30],
    'outcome_windows': [[0, 1], [0, 5], [1, 20]], 'minimum_estimation_days': 120,
    'expected_return_model': 'firm market model using CRSP value-weighted market return',
    'security_to_firm_aggregation': 'lagged-market-equity weighted PERMNO returns within PERMCO',
    'primary_sample': 'common 1,805-firm universe excluding every accepted seed firm',
    'primary_tests': ['Spearman score-CAR correlation', 'top-minus-bottom score quintile CAR',
                      'incremental adjusted R2 beyond frozen characteristics and SIC2 fixed effects'],
    'placebo': '100 frozen shuffled-seed all-S12 expansions',
    'multiple_testing': 'report channel/window/model grid transparently; no result-driven pruning',
    'timing_interpretation': 'researcher-information association test; not a tradable strategy',
    'expansion_manifest_sha256': freeze['sha256'], 'outcomes_inspected_before_freeze': False,
}
protocol['sha256'] = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()
(MANIFESTS / 'event_evaluation_protocol_v1.json').write_text(json.dumps(protocol, indent=2))
display(pd.DataFrame([protocol]).T.rename(columns={0: 'frozen_value'}))


,frozen_value
event_id,russia_ukraine_invasion_2022
event_date,2022-02-24
embedding_snapshot,2021Q4
estimation_window,"[-250, -30]"
outcome_windows,"[[0, 1], [0, 5], [1, 20]]"
minimum_estimation_days,120
expected_return_model,firm market model using CRSP value-weighted ma...
security_to_firm_aggregation,lagged-market-equity weighted PERMNO returns w...
primary_sample,"common 1,805-firm universe excluding every acc..."
primary_tests,"[Spearman score-CAR correlation, top-minus-bot..."


## 3. Optional standalone WRDS connection
Run this cell when any acquisition checkpoint is missing. It establishes and tests `db` independently. If all three input files already exist locally, you may skip this cell; Section 4 will not use WRDS.

In [ ]:
import wrds

username = input('WRDS username: ').strip()
password = getpass.getpass('WRDS password: ')

if 'db' in globals():
    try:
        db.close()
    except Exception:
        pass

db = wrds.Connection(
    wrds_username=username,
    wrds_password=password,
    autoconnect=False,
    verbose=True,
    wrds_connect_args={'sslmode': 'require', 'connect_timeout': 20,
                       'application_name': 'asset_embeddings_event_study'},
)
db.connect()  # Connect without automatically loading the full WRDS library catalog.
connection_test = db.raw_sql('select 1 as connection_test')
display(connection_test)
print('WRDS connection is active and queryable.')


postgresql://hermanthinnd:@wrds-pgdata.wharton.upenn.edu:9737/wrds
err=OperationalError('(psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: timeout expired\n')
postgresql://hermanthinnd:@wrds-pgdata.wharton.upenn.edu:9737/wrds
err=OperationalError('(psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: timeout expired\n')


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: timeout expired

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 4. Acquire daily returns, market returns, and point-in-time SIC
The cell probes standard CRSP schema variants, downloads only securities in the frozen universe, and checkpoints every extract. Rerunning it will load local Parquet files.

In [ ]:
daily_path = EVENT_DATA / 'crsp_daily_security_returns_2021_2022.parquet'
market_path = EVENT_DATA / 'crsp_value_weighted_market_2021_2022.parquet'
industry_path = EVENT_DATA / 'crsp_point_in_time_industry_2021q4.parquet'
needs_wrds = not all(path.exists() for path in [daily_path, market_path, industry_path])
if needs_wrds and ('db' not in globals() or getattr(db, 'connection', None) is None):
    raise ConnectionError('A checkpoint is missing. Run the standalone WRDS connection cell in Section 3 first.')

universe = pd.read_parquet(EXPANSION / 'text_only_scores.parquet')[['permco','firm_name']].drop_duplicates()
crosswalk = pd.read_parquet(ROOT / 'data/crosswalks/crsp/year=2021/quarter=4/stocknames_2021q4.parquet')
crosswalk = crosswalk[crosswalk.permco.isin(universe.permco)].copy()
permnos = sorted(crosswalk.permno.dropna().astype(int).unique())

def rollback_connection():
    try:
        db.connection.rollback()
    except Exception:
        pass

def first_queryable(candidates, query_builder):
    errors = []
    for candidate in candidates:
        try:
            probe = db.raw_sql(query_builder(candidate, True))
            return candidate
        except Exception as error:
            label = candidate['table'] if isinstance(candidate, dict) else candidate
            errors.append((label, type(error).__name__, str(error).splitlines()[0][:160]))
            rollback_connection()
    raise RuntimeError(f'No candidate table is queryable: {errors}')

if daily_path.exists():
    daily_security = pd.read_parquet(daily_path)
    daily_source = 'local_checkpoint'
else:
    daily_candidates = [
        {'table': 'crsp_a_stock.wrds_dsfv2_query', 'date': 'dlycaldt', 'ret': 'dlyret', 'prc': 'dlyprc', 'shrout': 'shrout'},
        {'table': 'crsp_a_stock.dsf_v2', 'date': 'dlycaldt', 'ret': 'dlyret', 'prc': 'dlyprc', 'shrout': 'shrout'},
        {'table': 'crsp_a_stock.dsf', 'date': 'date', 'ret': 'ret', 'prc': 'prc', 'shrout': 'shrout'},
        {'table': 'crsp.dsf', 'date': 'date', 'ret': 'ret', 'prc': 'prc', 'shrout': 'shrout'},
    ]
    daily_spec = first_queryable(daily_candidates, lambda s, probe: f"select permno, {s['date']}, {s['ret']}, {s['prc']}, {s['shrout']} from {s['table']} limit 1")
    daily_source = daily_spec['table']
    frames = []
    for start in range(0, len(permnos), 400):
        ids = ','.join(map(str, permnos[start:start+400]))
        frames.append(db.raw_sql(f"""select permno, {daily_spec['date']} as date,
            {daily_spec['ret']} as ret, {daily_spec['prc']} as prc, {daily_spec['shrout']} as shrout
            from {daily_source} where {daily_spec['date']} between '2021-02-01' and '2022-03-31'
            and permno in ({ids})""", date_cols=['date']))
        print(f'Daily-return batch {start//400 + 1} complete')
    daily_security = pd.concat(frames, ignore_index=True).merge(
        crosswalk[['permno','permco']].drop_duplicates(), on='permno', how='inner', validate='many_to_one')
    daily_security.to_parquet(daily_path, index=False)

if market_path.exists():
    market = pd.read_parquet(market_path)
    market_source = 'local_checkpoint'
else:
    market_candidates = ['crsp_a_indexes.dsi', 'crsp.dsi', 'crsp_a_stock.dsi']
    market_source = first_queryable(market_candidates, lambda t, probe: f"select date, vwretd from {t} limit 1")
    market = db.raw_sql(f"""select date, vwretd as market_return from {market_source}
        where date between '2021-02-01' and '2022-03-31'""", date_cols=['date'])
    market.to_parquet(market_path, index=False)

if industry_path.exists():
    industry = pd.read_parquet(industry_path)
    industry_source = 'local_checkpoint'
else:
    names_candidates = ['crsp_a_stock.stocknames', 'crsp.stocknames', 'crsp_a_stock.dsenames', 'crsp.dsenames']
    industry_source = first_queryable(names_candidates, lambda t, probe: f"select permno, permco, siccd, namedt, nameenddt from {t} limit 1")
    ids = ','.join(map(str, universe.permco.astype(int)))
    industry_detail = db.raw_sql(f"""select permno, permco, siccd, namedt, nameenddt from {industry_source}
        where permco in ({ids}) and namedt <= '2021-12-31' and nameenddt >= '2021-12-31'""",
        date_cols=['namedt','nameenddt'])
    industry_detail['siccd'] = pd.to_numeric(industry_detail.siccd, errors='coerce')
    industry = (industry_detail.dropna(subset=['siccd']).sort_values(['permco','permno'])
                .groupby('permco', as_index=False).first()[['permco','siccd']])
    industry['sic2'] = (industry.siccd // 100).astype('Int64')
    industry.to_parquet(industry_path, index=False)

acquisition_audit = pd.DataFrame([{'daily_source': daily_source, 'daily_rows': len(daily_security),
    'daily_firms': daily_security.permco.nunique(), 'market_source': market_source,
    'market_days': len(market), 'industry_source': industry_source,
    'industry_firms': industry.permco.nunique(), 'industry_coverage': universe.permco.isin(industry.permco).mean()}])
display(acquisition_audit)
acquisition_audit.to_parquet(EVENT_RESULTS / 'event_data_acquisition_audit.parquet', index=False)


## 5. Construct firm returns and frozen market-model CARs
Security returns are aggregated with lagged market equity. Firm-specific market models use trading days −250 through −30, and CARs use the three preregistered windows.

In [ ]:
firm_return_path = EVENT_DATA / 'daily_permco_returns.parquet'
car_path = EVENT_DATA / 'market_model_cars.parquet'
abnormal_path = EVENT_DATA / 'daily_abnormal_returns.parquet'
if firm_return_path.exists():
    firm_returns = pd.read_parquet(firm_return_path)
else:
    firm_returns = value_weight_firm_returns(daily_security)
    firm_returns.to_parquet(firm_return_path, index=False)
if car_path.exists() and abnormal_path.exists():
    cars, abnormal = pd.read_parquet(car_path), pd.read_parquet(abnormal_path)
else:
    cars, abnormal = market_model_cars(firm_returns, market, protocol['event_date'],
        tuple(protocol['estimation_window']), tuple(map(tuple, protocol['outcome_windows'])),
        protocol['minimum_estimation_days'])
    cars.to_parquet(car_path, index=False)
    abnormal.to_parquet(abnormal_path, index=False)
car_columns = ['car_0_1','car_0_5','car_1_20']
car_audit = pd.DataFrame([{'universe_firms': len(universe), 'firms_with_market_model': len(cars),
    **{f'{column}_coverage': cars[column].notna().mean() for column in car_columns},
    'median_estimation_observations': cars.estimation_observations.median()}])
display(car_audit)
display(cars[car_columns].describe(percentiles=[.01,.1,.5,.9,.99]).T)


## 6. Complete the industry benchmark and lock the evaluation panel
Every accepted seed firm is excluded from primary propagation tests. Direct-text results are retained separately as descriptive evidence because the accepted seed sample is only nine firms.

In [ ]:
seeds = pd.read_parquet(GEO_ROOT / 'documents/processed/text_exposure_seeds_frozen_v1.parquet')
universe_industry = universe.merge(industry[['permco','siccd','sic2']], on='permco', how='left', validate='one_to_one')
industry_scores = industry_seed_scores(universe_industry, seeds, 'sic2')
industry_scores.to_parquet(EVENT_DATA / 'industry_seed_scores.parquet', index=False)
embedding_scores = pd.read_parquet(EXPANSION / 'embedding_propagated_scores.parquet')
characteristic_scores = pd.read_parquet(EXPANSION / 'characteristic_propagated_scores.parquet')
model_scores = pd.concat([embedding_scores, characteristic_scores, industry_scores], ignore_index=True, sort=False)
evaluation = model_scores.loc[~model_scores.is_seed].merge(cars, on='permco', how='inner', validate='many_to_one')
evaluation.to_parquet(EVENT_DATA / 'locked_nonseed_evaluation_panel.parquet', index=False)
direct = seeds.merge(cars, left_on='firm_id', right_on='permco', how='left', validate='many_to_one')
direct.to_parquet(EVENT_RESULTS / 'direct_seed_event_reactions_descriptive.parquet', index=False)
display(evaluation.groupby(['representation','channel'], as_index=False).agg(firms=('permco','nunique')))
display(direct[['firm_id','channel','signed_score'] + car_columns])


## 7. Primary model comparison
For each representation, channel, and CAR window, report rank correlation, a top-minus-bottom quintile spread, and incremental adjusted R² beyond frozen characteristics plus SIC2 fixed effects. HC3 standard errors are used for the score coefficient.

In [ ]:
controls = pd.read_parquet(ROOT / 'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet')
control_columns = ['beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1']
firm_controls = universe_industry.merge(controls, on='permco', how='left', validate='one_to_one')
results = []
for (representation, channel), group in evaluation.groupby(['representation','channel'], sort=True):
    frame = (group.drop(columns=['sic2', 'siccd'], errors='ignore')
             .merge(firm_controls[['permco','sic2'] + control_columns],
                    on='permco', how='left', validate='many_to_one'))
    for column in control_columns:
        frame[column] = frame[column].fillna(frame[column].median())
    industry_dummies = pd.get_dummies(frame.sic2.astype('string').fillna('missing'), prefix='sic2', drop_first=True, dtype=float)
    base_x = pd.concat([frame[control_columns].reset_index(drop=True), industry_dummies.reset_index(drop=True)], axis=1)
    base_x = sm.add_constant(base_x.astype(float), has_constant='add')
    score = pd.to_numeric(frame.propagated_score, errors='coerce').astype('float64').reset_index(drop=True)
    score_sd = score.std(ddof=0)
    standardized_score = (score - score.mean()) / score_sd if score_sd > 0 else score * 0
    for outcome in car_columns:
        y = pd.to_numeric(frame[outcome], errors='coerce').astype('float64').reset_index(drop=True)
        valid = y.notna() & standardized_score.notna()
        rho, rho_p = spearmanr(score[valid], y[valid]) if score[valid].nunique() > 1 else (np.nan, np.nan)
        ranks = score[valid].rank(pct=True, method='average')
        low, high = y[valid][ranks <= .2], y[valid][ranks >= .8]
        spread = high.mean() - low.mean()
        spread_p = ttest_ind(high, low, equal_var=False, nan_policy='omit').pvalue if len(low) and len(high) else np.nan
        base = sm.OLS(y[valid], base_x.loc[valid]).fit(cov_type='HC3')
        augmented_x = base_x.loc[valid].copy(); augmented_x['exposure_score_z'] = standardized_score[valid]
        augmented = sm.OLS(y[valid], augmented_x).fit(cov_type='HC3')
        results.append({'representation': representation, 'channel': channel, 'outcome': outcome,
            'firms': int(valid.sum()), 'spearman_rho': rho, 'spearman_pvalue': rho_p,
            'top_minus_bottom_car': spread, 'spread_pvalue': spread_p,
            'score_coefficient': augmented.params['exposure_score_z'],
            'score_hc3_pvalue': augmented.pvalues['exposure_score_z'],
            'base_adjusted_r2': base.rsquared_adj, 'model_adjusted_r2': augmented.rsquared_adj,
            'incremental_adjusted_r2': augmented.rsquared_adj - base.rsquared_adj})
model_results = pd.DataFrame(results)
model_results.to_parquet(EVENT_RESULTS / 'locked_model_comparison.parquet', index=False)
display(model_results.sort_values(['outcome','channel','incremental_adjusted_r2'], ascending=[True,True,False]))


## 8. Shuffled-seed placebo test
The frozen all-S12 score is compared with the empirical distribution produced by 100 random assignments of identical seed weights. No placebo was selected after viewing returns.

In [ ]:
placebos = pd.read_parquet(EXPANSION / 'shuffled_embedding_placebos_100.parquet')
nonseed_ids = set(universe.permco) - set(seeds.firm_id)
placebos = placebos[placebos.permco.isin(nonseed_ids)].merge(cars[['permco'] + car_columns], on='permco', how='inner')
actual = model_results[model_results.representation.eq('all_s12')][['channel','outcome','spearman_rho']]
placebo_rows = []
for (repetition, channel), group in placebos.groupby(['repetition','channel'], sort=True):
    for outcome in car_columns:
        valid = group[['propagated_score', outcome]].dropna()
        rho = spearmanr(valid.propagated_score, valid[outcome]).statistic
        placebo_rows.append({'repetition': repetition, 'channel': channel, 'outcome': outcome, 'spearman_rho': rho})
placebo_correlations = pd.DataFrame(placebo_rows)
placebo_summary = actual.merge(
    placebo_correlations.groupby(['channel','outcome']).spearman_rho.agg(placebo_mean='mean', placebo_std='std').reset_index(),
    on=['channel','outcome'], how='left')
for index, row in placebo_summary.iterrows():
    null = placebo_correlations.query('channel == @row.channel and outcome == @row.outcome').spearman_rho.abs()
    placebo_summary.loc[index, 'empirical_two_sided_pvalue'] = (1 + (null >= abs(row.spearman_rho)).sum()) / (1 + len(null))
placebo_correlations.to_parquet(EVENT_RESULTS / 'placebo_correlations_100.parquet', index=False)
placebo_summary.to_parquet(EVENT_RESULTS / 'all_s12_placebo_summary.parquet', index=False)
display(placebo_summary)


## 9. Freeze evaluated outputs and decision limits
This final manifest hashes the acquired inputs and results. Statistical significance cannot repair weak channel identification: defense and oil each have one accepted seed, while the Russia–Ukraine channel has eight seeds of one direction.

In [ ]:
artifact_paths = [daily_path, market_path, industry_path, firm_return_path, car_path, abnormal_path,
    EVENT_DATA / 'industry_seed_scores.parquet', EVENT_DATA / 'locked_nonseed_evaluation_panel.parquet',
    EVENT_RESULTS / 'locked_model_comparison.parquet', EVENT_RESULTS / 'placebo_correlations_100.parquet',
    EVENT_RESULTS / 'all_s12_placebo_summary.parquet', EVENT_RESULTS / 'direct_seed_event_reactions_descriptive.parquet']
inventory = []
for path in artifact_paths:
    inventory.append({'path': str(path.relative_to(GEO_ROOT)), 'bytes': path.stat().st_size,
                      'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                      'rows': len(pd.read_parquet(path))})
inventory = pd.DataFrame(inventory)
inventory.to_parquet(EVENT_RESULTS / 'event_study_output_inventory.parquet', index=False)
final_manifest = {'stage': 'event_study_evaluated', 'protocol_sha256': protocol['sha256'],
    'expansion_manifest_sha256': freeze['sha256'], 'event_outcomes_opened': True,
    'interpretation': protocol['timing_interpretation'],
    'seed_limitations': {'defense_demand': 'one positive seed', 'oil_gas_production': 'one positive seed',
                         'russia_ukraine_operations_revenue': 'eight negative seeds'},
    'artifacts': inventory.to_dict('records')}
final_manifest['sha256'] = hashlib.sha256(json.dumps(final_manifest, sort_keys=True).encode()).hexdigest()
(MANIFESTS / 'event_study_evaluated_v1.json').write_text(json.dumps(final_manifest, indent=2))
display(inventory)
print('Final evaluated manifest SHA-256:', final_manifest['sha256'])
